# 36 · Append rows, Select rows and Chart, checked against pandas

## What this checks

This check follows one figure through three widgets. It covers **Chart** at release 0.25.2, **Append rows** at
release 0.6.0 and **Select rows** at release 0.5.1.

The figure is a lag-response plot. It shows how dengue cases relate to land surface temperature 0, 2, 4, 6 and 8
weeks earlier. Each lag has its own Poisson model, a regression for counts. Its result is a rate ratio: how many
times higher the case rate is when the temperature is higher.

- **Append rows** stacks the five models' tables into one, labelling each row with its lag in weeks.
- **Select rows** keeps the temperature rows.
- **Chart** draws each lag's rate ratio as a point, with its 95% interval as a whisker. The scale is logarithmic,
  with a dashed line at 1, where there is no effect.

A figure that shifts a point, or misplaces a lag, says something the models did not.

## How it is checked

We rebuild the same table with **pandas**, Python's library for tables, which shares no code with Fieldwork. pandas
stacks the five tables, keeps the temperature rows, and lines up each estimate with its two limits. Which number is
the estimate and which are the limits is read from the interval the data declare, as Fieldwork reads it.

The data are the five regression tables, the stacked table, and the points, scale and line that Chart's
specification draws.

## What should happen

1. **The same table**: every row of the stack, in order, has the same lag label, term, statistic and value.
2. **The same numbers**: at each lag, the point's estimate and both limits are the regression's, exactly. There is
   no tolerance, since nothing is computed between the model and the point.
3. **Lags as numbers**: the lags sit on a number axis, at 0, 2, 4, 6 and 8 weeks.
4. **A log scale**: its range holds every limit and 1, and the line is drawn at 1.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_36_lag_response import run

report = run(Path.cwd() / "fixtures" / "lag-response-brazil-001.json")
print("agrees:", report["agrees"])
print(f"rows: Fieldwork {report['rows']['fieldwork']}, pandas {report['rows']['pandas']}")
print()
for p in report["points"]:
    f = p["fieldwork"]
    print(f"  {'OK ' if p['agrees'] else 'DIFF'} lag {p['at']} weeks: rate ratio {f['estimate']} ({f['lower']} to {f['upper']})")
print()
for name, ok in report["checks"].items():
    print(f"  {'OK ' if ok else 'DIFF'} {name}")

## The result on record

First recorded on 7 November 2026, against Chart release 0.11.0. Rerun on 26 November 2026, against Chart release
0.25.2: **Agrees**.

The stacked table has 40 rows on both sides, the same row by row. Every point is exactly the regression's numbers. The lags sit on a number
axis, and the scale and line are as stated.

Running the cell above repeats the check on the same data.

## What this does not show

- That a lag-response plot is the right figure, or that any lag has an effect. Each point comes from its own model.
- That the intervals are right for weekly data. They allow nothing for one week being like the next.
- The drawn picture. This checks the instructions the chart is drawn from, not the pixels on screen.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])